# Notebook 01 — Preprocessing & Feature Engineering
**DSA 3900 | Maternal Health Risk Prediction — Kenya**

Loads the cleaned KDHS dataset, engineers WHO-based risk flags, builds the
composite risk score, performs a stratified 70/15/15 split, scales continuous
features and applies SMOTE to balance the training set.

In [1]:
# ── Libraries ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

print('Libraries loaded ✓')

Libraries loaded ✓


## 1. Load Cleaned Dataset

In [2]:
# Load the clean CSV produced by Notebook 00
DATA_IN  = Path('../../01_Data/processed/00_kdhs_clean.csv')
MODEL_OUT = Path('../../03_Models_Saved')
DATA_OUT  = Path('../../01_Data/processed')
MODEL_OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_IN)
print(f'Loaded: {df.shape}  columns: {list(df.columns)}')

Loaded: (10391, 24)  columns: ['v012', 'v201', 'v106', 'v190', 'v025', 'v501', 'v212', 'v024', 'm14_1', 'm15_1', 'm17_1', 'm3a_1', 'b11_01', 'f1_low_anc', 'f2_home_delivery', 'f3_no_doctor', 'f4_young_mother', 'f5_older_mother', 'f6_high_parity', 'f7_short_interval', 'f8_poor', 'risk_score', 'risk_class', 'risk_label']


## 2. Feature Selection

In [3]:
# 13 WHO-aligned features selected from literature review
FEAT_COLS = ['v012','v201','v106','v190','v025','v501','v212',
             'v024','m14_1','m15_1','m17_1','m3a_1','b11_01']

# Continuous columns that will be scaled
CONT_COLS = ['v012','v201','m14_1','b11_01','v212']

# Enforce numeric dtype (guard against object columns from pyreadstat)
for c in FEAT_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Impute ANC visits: code 98 = 'don\'t know' → NaN → median
df['m14_1'] = df['m14_1'].replace(98, np.nan)
df['m14_1'] = df['m14_1'].fillna(df['m14_1'].median())

print('Feature dtypes:')
print(df[FEAT_COLS].dtypes)

Feature dtypes:
v012        int64
v201        int64
v106        int64
v190        int64
v025        int64
v501        int64
v212      float64
v024        int64
m14_1     float64
m15_1     float64
m17_1     float64
m3a_1     float64
b11_01    float64
dtype: object


## 3. WHO-Based Risk Flag Engineering
Eight binary flags based on clinical risk factors identified in the literature.

In [4]:
# f1 – Adolescent mother (age at first birth ≤ 19)
df['f1'] = (df['v212'] <= 19).astype(int)

# f2 – High parity (≥ 5 previous births)
df['f2'] = (df['v201'] >= 5).astype(int)

# f3 – Low ANC attendance (< 4 visits, WHO minimum)
df['f3'] = (df['m14_1'] < 4).astype(int)

# f4 – Home delivery (place of delivery codes 11=home, 12=en route)
df['f4'] = df['m15_1'].isin([11, 12]).astype(int)

# f5 – No skilled attendant at delivery (m17_1 == 0)
df['f5'] = (df['m17_1'] == 0).astype(int)

# f6 – No doctor involved in ANC (m3a_1 == 0)
df['f6'] = (df['m3a_1'] == 0).astype(int)

# f7 – Short birth interval (< 24 months since last birth)
df['f7'] = (df['b11_01'] < 24).astype(int)

# f8 – Rural residence (v025 == 2)
df['f8'] = (df['v025'] == 2).astype(int)

FLAG_COLS = ['f1','f2','f3','f4','f5','f6','f7','f8']
print('Flag prevalence (%):')
print((df[FLAG_COLS].mean() * 100).round(1))

Flag prevalence (%):
f1    49.7
f2    22.5
f3    37.6
f4    18.2
f5    86.4
f6    57.5
f7    40.4
f8    65.4
dtype: float64


The flag distribution reveals the burden of maternal risk factors across the 10,391 Kenyan women with birth records:

| Flag | Description | Prevalence |
|------|-------------|-----------|
| f5 | No skilled delivery attendant | 86.4% |
| f8 | Rural residence | 65.4% |
| f6 | No doctor at ANC | 57.5% |
| f1 | Adolescent mother (first birth ≤19) | 49.7% |
| f7 | Short birth interval (<24 months) | 40.4% |
| f3 | Fewer than 4 ANC visits | 37.6% |
| f2 | High parity (≥5 births) | 22.5% |
| f4 | Home delivery | 18.2% |

The near-universal absence of skilled delivery attendants (f5) and high rural residence (f8)
point to systemic infrastructure gaps. Adolescent childbearing (f1) affecting half of women
underscores the role of socioeconomic factors in maternal risk. Home delivery (f4), while less
common, is clinically critical as it bypasses emergency obstetric care entirely.

## 4. Composite Risk Score & Target Variable

In [6]:
print(df[['f1','f2','f3','f4','f5','f6','f7','f8','risk_score']].describe())
print("\nRisk score distribution:")
print(df['risk_score'].value_counts().sort_index())

                 f1            f2            f3            f4            f5  \
count  10391.000000  10391.000000  10391.000000  10391.000000  10391.000000   
mean       0.496680      0.224906      0.375806      0.181888      0.863632   
std        0.500013      0.417541      0.484354      0.385771      0.343195   
min        0.000000      0.000000      0.000000      0.000000      0.000000   
25%        0.000000      0.000000      0.000000      0.000000      1.000000   
50%        0.000000      0.000000      0.000000      0.000000      1.000000   
75%        1.000000      0.000000      1.000000      0.000000      1.000000   
max        1.000000      1.000000      1.000000      1.000000      1.000000   

                 f6            f7            f8    risk_score  
count  10391.000000  10391.000000  10391.000000  10391.000000  
mean       0.574632      0.404100      0.653931      3.775575  
std        0.494422      0.490741      0.475738      1.661563  
min        0.000000      0.00000

In [5]:
# Composite score = sum of all 8 binary flags
df['risk_score'] = df[FLAG_COLS].sum(axis=1)

# 3-class target based on WHO risk stratification
# Low: 0–1 flags | Moderate: 2–3 flags | High: 4+ flags
def assign_class(score):
    if score <= 1:  return 0  # Low
    elif score <= 3: return 1  # Moderate
    else:           return 2  # High

df['risk_class'] = df['risk_score'].apply(assign_class)

dist = df['risk_class'].value_counts(normalize=True).sort_index() * 100
labels = {0:'Low', 1:'Moderate', 2:'High'}
for k,v in dist.items():
    print(f'  {labels[k]:10s}: {v:.1f}%')

  Low       : 8.1%
  Moderate  : 37.1%
  High      : 54.9%


The composite risk score (sum of 8 binary WHO-based flags) ranges from 0 to 8, with a median
of 4.0 — meaning the typical Kenyan woman in this dataset carries four simultaneous maternal
risk factors. The resulting class distribution reflects the genuine burden captured by KDHS 2022:

| Risk Class | Threshold | Share |
|------------|-----------|-------|
| Low        | 0–1 flags | 8.1%  |
| Moderate   | 2–3 flags | 37.1% |
| High       | 4+ flags  | 54.9% |

The dominance of the High Risk class (54.9%) is consistent with Kenya's structural maternal
health challenges: 86.4% of births occur without a skilled attendant, 65.4% of women reside
in rural areas with limited facility access, and over half receive ANC without physician
involvement. Only 8.1% of women escape all major risk factors simultaneously, underlining
the scale of unmet need in Kenya's maternal healthcare system. The severe class imbalance —
particularly the underrepresentation of Low Risk women — necessitates SMOTE oversampling
in the training pipeline to prevent model bias toward the majority class.

## 5. Train / Validation / Test Split  (70 / 15 / 15)

In [7]:
X = df[FEAT_COLS].copy()
y = df['risk_class'].copy()

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)

# Second split: 30% → 15% val + 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'Train : {X_train.shape}  |  Val : {X_val.shape}  |  Test : {X_test.shape}')
print('Class distribution in train:', dict(y_train.value_counts().sort_index()))

Train : (7273, 13)  |  Val : (1559, 13)  |  Test : (1559, 13)
Class distribution in train: {0: np.int64(588), 1: np.int64(2695), 2: np.int64(3990)}


The 10,391 records are partitioned using stratified sampling to preserve the original class
distribution across all three splits:

| Split      | Rows  | Low (0) | Moderate (1) | High (2) |
|------------|-------|---------|--------------|----------|
| Train (70%)| 7,273 | 588     | 2,695        | 3,990    |
| Val (15%)  | 1,559 | —       | —            | —        |
| Test (15%) | 1,559 | —       | —            | —        |

Stratification ensures each split mirrors the population distribution (8.1% / 37.1% / 54.9%),
preventing any split from being unrepresentative. The training set's severe imbalance —
particularly the Low Risk class with only 588 samples versus 3,990 High Risk — will be
corrected by SMOTE in the next step, which oversamples the minority classes to equality
without touching the validation or test sets.

## 6. Feature Scaling (StandardScaler — fit on train only)

In [8]:
scaler = StandardScaler()
X_train[CONT_COLS] = scaler.fit_transform(X_train[CONT_COLS])
X_val[CONT_COLS]   = scaler.transform(X_val[CONT_COLS])
X_test[CONT_COLS]  = scaler.transform(X_test[CONT_COLS])

joblib.dump(scaler, MODEL_OUT / 'scaler.pkl')
print('Scaler fitted and saved ✓')

Scaler fitted and saved ✓


A `StandardScaler` is fitted **exclusively on the training set** and applied to the validation
and test sets using the training statistics (mean and standard deviation). This strict protocol
prevents **data leakage** — a situation where information from unseen data influences the
model during training, leading to artificially optimistic performance estimates.

The five continuous features scaled are: age (`v012`), parity (`v201`), ANC visits (`m14_1`),
birth interval (`b11_01`), and age at first birth (`v212`). Scaling brings these features onto
a common range, which is essential for distance-based and gradient-based algorithms such as
Logistic Regression and TabNet. Tree-based models (Random Forest, XGBoost, CatBoost) are
scale-invariant but scaling them does not harm performance. The fitted scaler is saved to
`03_Models_Saved/scaler.pkl` for use in the deployment application.

## 7. SMOTE — Oversample Training Set Only

In [9]:
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE: {X_train.shape[0]} rows')
print(f'After  SMOTE: {X_train_sm.shape[0]} rows')
print('Class counts after SMOTE:', dict(pd.Series(y_train_sm).value_counts().sort_index()))

Before SMOTE: 7273 rows
After  SMOTE: 11970 rows
Class counts after SMOTE: {0: np.int64(3990), 1: np.int64(3990), 2: np.int64(3990)}


SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic samples for the
minority classes by interpolating between existing observations in feature space, using
k=5 nearest neighbours. It is applied **only to the training set**, leaving the validation
and test sets in their natural distribution for unbiased evaluation.

| Class    | Before SMOTE | After SMOTE |
|----------|-------------|-------------|
| Low      | 588         | 3,990       |
| Moderate | 2,695       | 3,990       |
| High     | 3,990       | 3,990       |
| **Total**| **7,273**   | **11,970**  |

The Low Risk class grew by 6.8× and Moderate by 1.5×, bringing all three classes to
perfect balance at 3,990 samples each. This prevents the model from defaulting to
predicting the majority class (High Risk) and ensures it learns meaningful decision
boundaries for all three risk levels — particularly critical for the clinically important
High Risk class where false negatives carry the greatest patient safety consequences.

## 8. Save Splits

In [10]:
# Save all splits to processed folder
X_train_sm.to_csv(DATA_OUT / 'X_train.csv', index=False)
X_val.to_csv(DATA_OUT / 'X_val.csv', index=False)
X_test.to_csv(DATA_OUT / 'X_test.csv', index=False)
pd.Series(y_train_sm, name='risk_class').to_csv(DATA_OUT / 'y_train.csv', index=False)
pd.Series(y_val,      name='risk_class').to_csv(DATA_OUT / 'y_val.csv',   index=False)
pd.Series(y_test,     name='risk_class').to_csv(DATA_OUT / 'y_test.csv',  index=False)

print('All splits saved ✓')
print(f'  X_train : {X_train_sm.shape}')
print(f'  X_val   : {X_val.shape}')
print(f'  X_test  : {X_test.shape}')

All splits saved ✓
  X_train : (11970, 13)
  X_val   : (1559, 13)
  X_test  : (1559, 13)


## ✅ Preprocessing Complete

| Split | Rows   | Note |
|-------|--------|------|
| Train | 11,970 | SMOTE-balanced (3,990 per class) |
| Val   | 1,559  | Real distribution |
| Test  | 1,559  | Held out — unseen until Notebook 05 |